# ComfyUI on Kaggle (LTX Video + VAE-only Decode)

**Setup in Kaggle Notebook Settings:**
1. **Accelerator: GPU T4 x2** (oder P100)
2. **Internet: ON**
3. **Add Dataset:** ein Dataset mit deinen VAE-Dateien (`.safetensors` / `.ckpt` / `.pt` / `.bin`).
   Es wird unter `/kaggle/input/<dataset-name>/` gemountet (read-only).

- Kein Google Drive noetig - ComfyUI und Modelle liegen auf der persistenten Disk `/kaggle/working`.
- LTX-Video Checkpoints/Diffusion-Modelle muessen selbst in `/kaggle/working/ComfyUI/models/...` gelegt werden (Upload als Session-Datei oder Kaggle Dataset + Copy).
- Es wird nichts neu installiert, wenn ComfyUI schon existiert - nur `git pull`.
- Tunnel: Zelle 4 (Cloudflare) ODER Zelle 5 (Pinggy) ausfuehren.

In [ ]:
# Zelle 1: cloudflared Binary holen
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Zelle 2: ComfyUI Setup (git clone / git pull), Custom Nodes, VAEs aus Kaggle-Dataset verlinken
import os
import shlex
import subprocess
import threading
import queue
import glob

ComfyUIRoot = "/kaggle/working/ComfyUI"
port = 8188  # autoritative Port-Setting
Extra_arguments = "--listen --enable-manager"  # kein Port hier

# --- Kaggle-Dataset mit VAEs finden (Add Dataset -> /kaggle/input/<name>/) ---
vae_extensions = (".safetensors", ".ckpt", ".pt", ".bin")
vae_drive_path = None
for candidate in sorted(glob.glob("/kaggle/input/*")):
    hits = [f for f in glob.glob(os.path.join(candidate, "**", "*"), recursive=True)
            if f.lower().endswith(vae_extensions)]
    if hits:
        vae_drive_path = candidate
        break

if vae_drive_path:
    print(f"VAE-Dataset gefunden: {vae_drive_path}")
else:
    print("WARNUNG: Kein Dataset mit VAE-Dateien unter /kaggle/input gefunden.")
    print("Bitte 'Add Dataset' im rechten Panel nutzen (VAEs .safetensors/.ckpt/.pt/.bin).")

# --- Zeige gefundene VAEs auf Dataset ---
if vae_drive_path:
    found_vaes = [f for f in glob.glob(os.path.join(vae_drive_path, "**", "*"), recursive=True)
                  if f.lower().endswith(vae_extensions)]
    print(f"Gefundene VAEs ({len(found_vaes)}):")
    for f in found_vaes:
        print("  ", os.path.relpath(f, vae_drive_path))

# --- ComfyUI: vorhanden -> git pull, sonst git clone (KEINE Neuinstallation per ZIP) ---
def update_repo(repo_url, target_dir):
    if os.path.isdir(os.path.join(target_dir, ".git")):
        print(f"Update via git pull: {target_dir}")
        r = subprocess.run(["git", "-C", target_dir, "pull"], capture_output=True, text=True)
        if r.returncode != 0:
            print("Pull-Konflikt, versuche reset + pull...")
            subprocess.run(["git", "-C", target_dir, "checkout", "."], capture_output=True)
            subprocess.run(["git", "-C", target_dir, "pull", "--rebase"], capture_output=True)
        print(r.stdout.strip() or "(up to date)")
    else:
        print(f"Clone: {repo_url} -> {target_dir}")
        subprocess.run(["git", "clone", "--depth", "1", repo_url, target_dir], check=True)

update_repo("https://github.com/comfyanonymous/ComfyUI.git", ComfyUIRoot)

# --- Geforderte Repositories installieren ---
custom_nodes_dir = os.path.join(ComfyUIRoot, "custom_nodes")
os.makedirs(custom_nodes_dir, exist_ok=True)

def install_custom_node(repo_url):
    name = repo_url.rstrip("/").split("/")[-1].replace(".git", "")
    update_repo(repo_url, os.path.join(custom_nodes_dir, name))

install_custom_node('https://github.com/Comfy-Org/ComfyUI-Manager.git')
install_custom_node('https://github.com/Lightricks/ComfyUI-LTXVideo.git')
install_custom_node('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git')
install_custom_node('https://github.com/DemonAlone/DemonAlone-nodes-ComfyUI.git')

# --- Requirements installieren: Abbruch bei Fehler (check=True) ---
subprocess.run(
    ["pip", "install", "-r", os.path.join(ComfyUIRoot, "requirements.txt")],
    check=True,
)

# --- Custom-Node-Requirements (best effort, Server soll nicht blockieren) ---
for req in glob.glob(os.path.join(custom_nodes_dir, "*", "requirements.txt")):
    subprocess.run(["pip", "install", "-r", req])

# --- VAEs per Symlink aus dem Kaggle-Dataset in ComfyUI/models/vae verlinken ---
vae_local = os.path.join(ComfyUIRoot, "models", "vae")
os.makedirs(vae_local, exist_ok=True)
if vae_drive_path:
    linked = 0
    for f in glob.glob(os.path.join(vae_drive_path, "**", "*"), recursive=True):
        if f.lower().endswith(vae_extensions) and os.path.isfile(f):
            dst = os.path.join(vae_local, os.path.basename(f))
            if not os.path.exists(dst):
                try:
                    os.symlink(f, dst)
                    linked += 1
                    print("Symlink:", os.path.basename(f))
                except OSError as e:
                    import shutil
                    shutil.copy2(f, dst)  # Fallback: Dataset ist read-only, Symlink moeglich
                    linked += 1
                    print("Copy:", os.path.basename(f))
    print(f"{linked} VAE(s) verlinkt nach {vae_local}")

# --- ComfyUI starten (Port via --port, shlex.split fuer korrekte Quote-Handles) ---
cmd = ["python", os.path.join(ComfyUIRoot, "main.py"), "--port", str(port)]
cmd += shlex.split(Extra_arguments)
comfy_proc = subprocess.Popen(cmd, cwd=ComfyUIRoot)
print("ComfyUI gestartet, PID:", comfy_proc.pid)

In [ ]:
# Zelle 3 (Option A): Cloudflare-Tunnel (trycloudflare, ohne Account)
import subprocess, threading, queue, time, re

output_queue = queue.Queue()

def get_free_port(start=8200):
    import socket
    for p in range(start, start + 100):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", p)) != 0:
                return p
    raise RuntimeError("Kein freier Metrics-Port gefunden")

metrics_port = get_free_port()

def cloudflared_worker():
    try:
        proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}",
             "--metrics", f"127.0.0.1:{metrics_port}"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.STDOUT,
        )
        url = None
        deadline = time.time() + 60
        while url is None and time.time() < deadline:
            try:
                import urllib.request, json
                with urllib.request.urlopen(f"http://127.0.0.1:{metrics_port}/routes", timeout=5) as resp:
                    data = json.loads(resp.read().decode())
                    for entry in data:
                        if isinstance(entry, dict) and "public_hostname" in entry:
                            url = "https://" + entry["public_hostname"]
                            break
            except Exception:
                pass
            if proc.poll() is not None:
                raise RuntimeError(f"cloudflared beendet (rc={proc.returncode})")
            time.sleep(2)
        if url is None:
            raise RuntimeError("Tunnel-URL nicht innerhalb 60s gefunden")
        output_queue.put(("ok", url))
    except Exception as e:
        output_queue.put(("error", e))

thread = threading.Thread(target=cloudflared_worker, daemon=True)
thread.start()
thread.join(timeout=90)

kind, payload = output_queue.get_nowait()
if kind == "error":
    raise payload
tunnel_url = payload
print("=" * 60)
print("ComfyUI Cloudflare Tunnel (anonym, kein Login):")
print(tunnel_url)
print("=" * 60)

In [ ]:
# Zelle 4 (Option B): Pinggy-Tunnel - alternativ zu Zelle 3
# Windows-Batch-Logik als Reconnect-Loop:
#   FOR /L %N IN () DO ( ssh ... @free.pinggy.io & timeout /t 10 )
import subprocess, threading, queue, re, time

PINGGY_TOKEN = "DphNQlI9bNG"  # eigenen Pinggy-Token hier eintragen
pinggy_queue = queue.Queue()

def pinggy_worker():
    cmd = [
        "ssh", "-p", "443",
        "-R", f"0:127.0.0.1:{port}",
        "-o", "StrictHostKeyChecking=no",
        "-o", "ServerAliveInterval=30",
        f"{PINGGY_TOKEN}@free.pinggy.io",
    ]
    while True:
        try:
            proc = subprocess.Popen(
                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1,
            )
            assert proc.stdout is not None
            for line in proc.stdout:
                print(line.rstrip())
                m = re.search(r"https://\S+\.pinggy\.io\S*", line)
                if m:
                    pinggy_queue.put(m.group(0).rstrip("."))
            rc = proc.wait()
            print(f"Pinggy getrennt (rc={rc}), neuer Versuch in 10s...")
        except Exception as e:
            print("Pinggy-Fehler:", e, "- neuer Versuch in 10s...")
        time.sleep(10)

t = threading.Thread(target=pinggy_worker, daemon=True)
t.start()

try:
    tunnel_url = pinggy_queue.get(timeout=60)
    print("=" * 60)
    print("ComfyUI Pinggy Tunnel:")
    print(tunnel_url)
    print("=" * 60)
except queue.Empty:
    print("Keine Pinggy-URL innerhalb 60s erhalten - Token pruefen!")

In [ ]:
# Zelle 5: Ausgaben als ZIP exportieren (Kaggle Download ueber 'Output' Tab)
import os, zipfile, glob

drive_root = "/kaggle/working"  # Kaggle persistenter Output-Ordner
zip_subfolder = "outputs"      # Unterordner innerhalb von drive_root
base_path = os.path.join(drive_root, zip_subfolder)

media_exts = (".mp4", ".png", ".webm")
candidates = []
for ext in media_exts:
    candidates += glob.glob(os.path.join(base_path, "**", f"*{ext}"), recursive=True)

# Deduplizieren und nur echte Dateien beruecksichtigen
files = sorted({f for f in candidates if os.path.isfile(f)})

zip_name = os.path.join(drive_root, "comfyui_outputs.zip")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files:
        zf.write(f, arcname=os.path.relpath(f, base_path))
        print("Hinzugefuegt:", os.path.relpath(f, base_path))

print(f"ZIP erstellt: {zip_name} ({len(files)} Datei(en))")
print("Download ueber die 'Output'-Registerkarte des Notebooks.")